# 27 · Are there hidden batting types? Features first, then clustering

> **Run on the first data file.** This research notebook and its saved outputs come from the ball-by-ball file as first supplied (every extra counted as 1 run). On 16 Nov 2026 the organisers issued a corrected file with the real extra runs; the method was re-run and re-checked on it in [notebook 28](28_final_review_corrected_data.ipynb), and both top-10 lists kept the same names.

1. **Features first.** 22 candidate style features in six families: outcomes against expected, scoring shape, tempo, shot mix (from the stroke column), match-ups and phase.
2. **Feature selection.** Keep only stable traits (similar in two halves of a player's matches), remove the effect of the batting job, drop near-duplicates: 13 features remain.
3. **Seven algorithms.** k-means, Ward, spectral, Gaussian mixture (BIC may choose one group), Bayesian mixture, HDBSCAN, UMAP + HDBSCAN.
4. **A fair test.** Each must beat 50 copies of the data with the same features, skew and correlations but no groups, and give the same groups from two halves of the matches. As a check that the test works, it is first run on the job features, where groups really exist.

**Result.** The test finds the job groups easily, but **no playing-style types**: no algorithm beats the no-group copies, the Gaussian mixture chooses one group, HDBSCAN finds none, and groups from two halves of the matches do not agree. Style is a set of sliding scales, not types. Three scales explain 64% of the differences: attacker v accumulator, dot-tolerant v always scoring, classical v unorthodox. Speeding up later in an innings (the anchor's trademark) is not a repeatable trait in this data.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes and the rule fixed before running</summary>

P30 - Are there hidden batting types? Features first, then algorithms, then a test against 'no types' (research).  
  
User's point (2026-11-15): choose the features first, then the clustering method; k-means alone is not enough.  
Everything below is fixed BEFORE running.  
Population: batters with 200+ normal (non-pressure) balls faced. Only normal balls are used, so the pressure balls being  
ranked never define a type.  
STEP 1 - candidate features (25), six families, each a per-ball average so noise can be measured the same way:  
  outcome v expected (power, dots, risk); scoring shape (six share of boundaries, rotation, hard running, off-side share);  
  tempo (start strike rate, start dots, acceleration = balls 20+ minus balls 1-9); shot mix (slog/innovative, drives,  
  cross-bat, nudges, defence, sweeps v spin); match-ups (power v spin minus v pace, dots v spin minus v pace, power v short  
  balls, v full balls, v slower balls); phase (power at the death minus in the middle overs).  
STEP 2 - feature selection (no target exists, so the rules are about being a stable trait, not about the ranking):  
  (a) reliability: split-half correlation across matches (20 random halvings, Spearman-Brown corrected) >= 0.5;  
  (b) each player's value shrunk toward the average by its own noise (empirical Bayes);  
  (c) the job removed: each feature regressed on usage (position shares, entry over, phase shares) -> style within job;  
  (d) redundancy: of any pair with |Spearman| > 0.8, the less reliable one is dropped.  
STEP 3 - algorithms: k-means, Gaussian mixture (BIC chooses k, k = 1 allowed), Bayesian (Dirichlet) mixture, Ward,  
  spectral, HDBSCAN, UMAP + HDBSCAN.  
STEP 4 - test against 'no types': 50 null datasets with the same features, same skew and same correlations but no  
  groups (Gaussian copula). Positive control: the same test on the USAGE features (the job), where groups should exist.  
  Real types exist only if an algorithm beats ALL 50 nulls (p <= 0.02) AND gives the same groups from two halves of the  
  matches (adjusted Rand >= 0.5). Otherwise style is reported as continuous axes (PCA), not types.  
Sequential, 2 threads, low priority.
</details>

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["NUMBA_NUM_THREADS"] = "2"
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering, HDBSCAN
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from rrlib.common import SEED, load, load_table, save_table, save_json, get_logger, Timer, excluded_reasons
try:
    import psutil
    psutil.Process().nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
except Exception:
    pass
import umap

log = get_logger("p30_style_discovery")
pd.set_option("display.width", 230)
MIN_BALLS, N_NULL, N_NULL_UMAP = 200, 50, 20
v = load("values", columns=["ball_uid", "match_id", "innings", "striker_id", "batting_order", "phase", "is_pace", "is_pressure",
                            "faced", "runs", "boundary", "r_bdry_bat", "r_dot_bat", "r_wkt_bat"])
bs = load("ball_state", columns=["ball_uid", "stroke", "delivery_style", "on_or_off", "str_balls", "is_six", "legal_before"])
v = v.merge(bs, on="ball_uid", how="left")
d = v[v.faced.astype(bool) & ~v.is_pressure.astype(bool)].copy()
cnt = d.groupby("striker_id").size()
POP = cnt.index[cnt >= MIN_BALLS]
d = d[d.striker_id.isin(POP)]
log.info(f"population: {len(POP)} batters with {MIN_BALLS}+ normal balls; {len(d):,} balls")

# ---------------------------------------------------------------- STEP 1: per-ball candidate features
SLOG = {"Slog", "Slog Sweep", "PickUp", "Inside out", "UpperCut", "Paddle Scoop", "Paddle Sweep", "Reverse Paddle Scoop",
        "Reverse Sweep", "Switch"}
DRIVE = {"Cover Drive", "Off Drive", "On drive", "Straight Drive", "Square drive"}
CROSS = {"Pull shot", "Hook Shot", "Square Cut", "Late Cut"}
NUDGE = {"Worked", "Flick", "Leg Glance", "Glance", "Steer", "Push"}
DEF = {"Frontfoot defensive", "Backfoot defensive", "Defense", "Intentional Padding"}
SWEEP = {"Sweep Shot", "Slog Sweep", "Reverse Sweep", "Paddle Sweep"}
SHORT = {"Bouncer", "Slow Bouncer"}
FULL = {"Yorker", "Wide Yorker", "Full Toss"}
SLOWER = {"Off Cutter", "Leg Cutter", "Knuckle", "Slow Bouncer"}


def ball_features(d):
    sk = d.stroke.notna()
    spin = ~d.is_pace.astype(bool)
    f = pd.DataFrame(index=d.index)
    f["power"], f["dots"], f["risk"] = d.r_bdry_bat, d.r_dot_bat, d.r_wkt_bat
    f["six_share"] = d.is_six.astype(float).where(d.boundary == 1)
    f["rotation"] = d.runs.between(1, 3).astype(float)
    f["hard_running"] = d.runs.isin([2, 3]).astype(float).where(d.runs.between(1, 3))
    f["off_side"] = (d.on_or_off == "Off-side").astype(float).where(d.runs > 0)
    f["start_sr"] = d.runs.where(d.str_balls < 10).astype(float)
    f["start_dots"] = (d.runs == 0).astype(float).where(d.str_balls < 10)
    f["_late_sr"] = d.runs.where(d.str_balls >= 20).astype(float)
    for nm, S in (("shot_slog", SLOG), ("shot_drive", DRIVE), ("shot_cross", CROSS), ("shot_nudge", NUDGE), ("shot_defence", DEF)):
        f[nm] = d.stroke.isin(S).astype(float).where(sk)
    f["sweep_v_spin"] = d.stroke.isin(SWEEP).astype(float).where(sk & spin)
    f["_power_spin"], f["_power_pace"] = d.r_bdry_bat.where(spin), d.r_bdry_bat.where(~spin)
    f["_dots_spin"], f["_dots_pace"] = d.r_dot_bat.where(spin), d.r_dot_bat.where(~spin)
    f["power_v_short"] = d.r_bdry_bat.where(d.delivery_style.isin(SHORT))
    f["power_v_full"] = d.r_bdry_bat.where(d.delivery_style.isin(FULL))
    f["power_v_slower"] = d.r_bdry_bat.where(d.delivery_style.isin(SLOWER))
    f["_power_death"], f["_power_mid"] = d.r_bdry_bat.where(d.phase == 2), d.r_bdry_bat.where(d.phase == 1)
    return f


DIFFS = {"acceleration": ("_late_sr", "start_sr"), "spin_minus_pace_power": ("_power_spin", "_power_pace"),
         "spin_minus_pace_dots": ("_dots_spin", "_dots_pace"), "death_minus_middle_power": ("_power_death", "_power_mid")}


def player_table(d):
    """per player: mean, sampling variance of the mean, for every candidate feature (direct + differences)"""
    f = ball_features(d)
    g = f.groupby(d.striker_id.values)
    m, c, s2 = g.mean(), g.count(), g.var()
    se2 = s2 / c.replace(0, np.nan)
    M, SE2 = {}, {}
    for col in [x for x in f.columns if not x.startswith("_")]:
        M[col], SE2[col] = m[col], se2[col]
    for nm, (a, b) in DIFFS.items():
        M[nm], SE2[nm] = m[a] - m[b], se2[a] + se2[b]
    return pd.DataFrame(M), pd.DataFrame(SE2), c


def shrink(M, SE2):
    """empirical Bayes: pull each player toward the average in proportion to his own noise"""
    out = M.copy()
    for col in M.columns:
        x, e = M[col], SE2[col].fillna(SE2[col].max())
        mu = np.average(x.dropna(), weights=1 / e[x.notna()].clip(1e-9))
        tau2 = max(np.nanvar(x) - np.nanmean(e), 1e-12)
        out[col] = mu + (x.fillna(mu) - mu) * (tau2 / (tau2 + e))
    return out


M, SE2, C = player_table(d)
CAND = list(M.columns)
log.info(f"{len(CAND)} candidate features: {CAND}")

# ---------------------------------------------------------------- STEP 2a: reliability (split-half across matches)
rng = np.random.RandomState(SEED)
mids = d.match_id.unique()
rel = {c: [] for c in CAND}
with Timer(log, "split-half reliability, 20 halvings"):
    for _ in range(20):
        A = set(rng.choice(mids, len(mids) // 2, replace=False)); ma = d.match_id.isin(A)
        Ma, _, Ca = player_table(d[ma]); Mb, _, Cb = player_table(d[~ma])
        ix = Ma.index.intersection(Mb.index)
        ix = ix[(d[ma].groupby("striker_id").size().reindex(ix) >= 60).values & (d[~ma].groupby("striker_id").size().reindex(ix) >= 60).values]
        for c in CAND:
            a, b = Ma.loc[ix, c], Mb.loc[ix, c]
            ok = a.notna() & b.notna()
            r = np.corrcoef(a[ok], b[ok])[0, 1] if ok.sum() > 30 else np.nan
            rel[c].append(2 * r / (1 + r) if r == r and r > -0.99 else np.nan)
REL = pd.Series({c: np.nanmean(x) for c, x in rel.items()}).sort_values(ascending=False)
KEEP1 = REL.index[REL >= 0.5].tolist()

# ---------------------------------------------------------------- STEP 2b-d: shrink, remove the job, drop duplicates
S = shrink(M, SE2)
fb = v[v.faced.astype(bool) & (v.innings <= 2) & v.striker_id.isin(POP)]
pos_inn = fb.groupby(["striker_id", "match_id"]).batting_order.first()
U = pd.crosstab(pos_inn.index.get_level_values(0), pd.cut(pos_inn.values, [0, 2, 4, 6, 11], labels=["pos_1_2", "pos_3_4", "pos_5_6", "pos_7_11"]),
                normalize="index")
U.columns = U.columns.astype(str)
U["entry_over"] = fb[fb.str_balls == 0].groupby("striker_id").legal_before.median() / 6 + 1
for k_, ph in (("pp_share", 0), ("mid_share", 1), ("death_share", 2)):
    U[k_] = (fb.phase == ph).groupby(fb.striker_id).mean()
U = U.reindex(POP).fillna(U.median())
JOB = ["pos_1_2", "pos_3_4", "pos_5_6", "entry_over", "pp_share", "death_share"]
R2job, Sres = {}, pd.DataFrame(index=POP)
for c in CAND:
    lr = LinearRegression().fit(U[JOB], S.loc[POP, c])
    R2job[c] = lr.score(U[JOB], S.loc[POP, c])
    Sres[c] = S.loc[POP, c] - lr.predict(U[JOB])
FT = pd.DataFrame({"reliability": REL, "explained_by_job_R2": pd.Series(R2job)}).sort_values("reliability", ascending=False)
FT["kept_reliable"] = FT.index.isin(KEEP1)
cor = Sres[KEEP1].corr(method="spearman").abs()
SEL = []
for c in sorted(KEEP1, key=lambda x: -REL[x]):
    if all(cor.loc[c, s_] <= 0.8 for s_ in SEL):
        SEL.append(c)
FT["selected"] = FT.index.isin(SEL)
save_table(FT.reset_index().rename(columns={"index": "feature"}), "style_feature_selection")
log.info("feature selection:\n" + FT.round(2).to_string())
log.info(f"selected {len(SEL)} features: {SEL}")


# ---------------------------------------------------------------- STEP 3-4: algorithms v nulls
def algos(X, with_umap):
    out = {}
    for k in range(2, 7):
        out[f"k-means k={k}"] = KMeans(k, n_init=10, random_state=SEED).fit_predict(X)
        out[f"Ward k={k}"] = AgglomerativeClustering(k, linkage="ward").fit_predict(X)
        out[f"spectral k={k}"] = SpectralClustering(k, affinity="nearest_neighbors", n_neighbors=10, random_state=SEED,
                                                    assign_labels="cluster_qr").fit_predict(X)
    bics = {k: GaussianMixture(k, covariance_type="full", n_init=3, random_state=SEED).fit(X).bic(X) for k in range(1, 7)}
    kb = min(bics, key=bics.get)
    out["GMM (BIC picks k)"] = GaussianMixture(kb, covariance_type="full", n_init=3, random_state=SEED).fit_predict(X) if kb > 1 else np.zeros(len(X), int)
    bg = BayesianGaussianMixture(n_components=8, weight_concentration_prior=0.1, n_init=2, random_state=SEED, max_iter=500).fit(X)
    out["Bayesian mixture"] = bg.predict(X)
    out["HDBSCAN"] = HDBSCAN(min_cluster_size=10).fit_predict(X)
    if with_umap:
        E = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.0, random_state=SEED, n_jobs=1).fit_transform(X)
        out["UMAP + HDBSCAN"] = HDBSCAN(min_cluster_size=10).fit_predict(E)
    return out


def sil(X, lab):
    m = lab >= 0
    if m.sum() < 10 or len(np.unique(lab[m])) < 2:
        return np.nan
    return silhouette_score(X[m], lab[m])


def describe(lab):
    m = lab >= 0
    return int(len(np.unique(lab[m]))), float(1 - m.mean())


def null_like(X, rs):
    """Gaussian copula: same marginals (skew), same rank correlations, no groups"""
    rho = pd.DataFrame(X).corr(method="spearman").values
    P = 2 * np.sin(np.pi * rho / 6)
    Z = rs.multivariate_normal(np.zeros(X.shape[1]), P, size=len(X))
    out = np.empty_like(X)
    for j in range(X.shape[1]):
        out[np.argsort(Z[:, j]), j] = np.sort(X[:, j])
    return out


def test_structure(X, label, n_null, n_null_umap):
    real = algos(X, True)
    res = {nm: {"set": label, "algorithm": nm, "groups": describe(l)[0], "noise_share": describe(l)[1], "silhouette": sil(X, l),
                "null_sil": []} for nm, l in real.items()}
    rs = np.random.RandomState(SEED + 7)
    for i in range(n_null):
        Xn = null_like(X, rs)
        for nm, l in algos(Xn, i < n_null_umap).items():
            res[nm]["null_sil"].append(sil(Xn, l))
    rows = []
    for nm, r in res.items():
        ns = np.array(r.pop("null_sil"), float)
        ns_ok = np.nan_to_num(ns, nan=-1.0)
        r["null_sil_median"] = float(np.nanmedian(ns)) if np.isfinite(ns).any() else np.nan
        r["null_sil_max"] = float(np.nanmax(ns)) if np.isfinite(ns).any() else np.nan
        r["p_value"] = float((1 + (ns_ok >= r["silhouette"]).sum()) / (1 + len(ns))) if r["silhouette"] == r["silhouette"] else np.nan   # no groups -> no test
        rows.append(r)
    return pd.DataFrame(rows), real


Xs = StandardScaler().fit_transform(Sres[SEL].values)
Xu = StandardScaler().fit_transform(U[JOB].values)
with Timer(log, "positive control: job (usage) features"):
    T_job, real_job = test_structure(Xu, "job (positive control)", N_NULL, N_NULL_UMAP)
with Timer(log, "playing style features"):
    T_sty, real_sty = test_structure(Xs, "playing style", N_NULL, N_NULL_UMAP)
T = pd.concat([T_job, T_sty], ignore_index=True)


# ---------------------------------------------------------------- stability: two halves of the matches -> same groups?
def style_matrix(dd):
    Mh, SEh, _ = player_table(dd)
    Sh = shrink(Mh, SEh).reindex(POP)
    out = pd.DataFrame(index=POP)
    for c in SEL:
        x = Sh[c].fillna(Sh[c].mean())
        out[c] = x - LinearRegression().fit(U[JOB], x).predict(U[JOB])
    return out


rs2 = np.random.RandomState(SEED + 3)
halves = []
for _ in range(5):
    A = set(rs2.choice(mids, len(mids) // 2, replace=False))
    halves.append((style_matrix(d[d.match_id.isin(A)]), style_matrix(d[~d.match_id.isin(A)]),
                   d[d.match_id.isin(A)].groupby("striker_id").size(), d[~d.match_id.isin(A)].groupby("striker_id").size()))
ari = {nm: [] for nm in real_sty}
sc_ = StandardScaler().fit(Sres[SEL].values)
with Timer(log, "split-half stability of the groups"):
    for Ha, Hb, ca, cb in halves:
        ix = POP[(ca.reindex(POP).fillna(0) >= 80).values & (cb.reindex(POP).fillna(0) >= 80).values]
        la, lb = algos(sc_.transform(Ha.loc[ix].values), True), algos(sc_.transform(Hb.loc[ix].values), True)
        for nm in real_sty:
            ari[nm].append(adjusted_rand_score(la[nm], lb[nm]))
ari = {nm: float(np.mean(x)) for nm, x in ari.items()}
T["split_half_ARI"] = T.algorithm.map(ari).where(T.set == "playing style")
T["real_types"] = (T.groups >= 2) & (T.p_value <= 0.02) & (T.split_half_ARI >= 0.5)
save_table(T, "style_cluster_tests")
log.info("\n" + T.round(3).to_string(index=False))

# ---------------------------------------------------------------- continuous style axes (PCA) + top-10 profiles
pca = PCA().fit(Xs)
ev = pca.explained_variance_ratio_
L = pd.DataFrame(pca.components_[:4].T, index=SEL, columns=[f"axis{i + 1}" for i in range(4)])
log.info(f"PCA variance explained: {np.round(ev[:6], 3).tolist()} (cumulative 3 axes {ev[:3].sum():.2f})")
log.info("axis loadings:\n" + L.round(2).to_string())
save_table(L.reset_index().rename(columns={"index": "feature"}), "style_axes_loadings")
EX = set(excluded_reasons()); names = load("players").set_index("player_id").name
top = load_table("final_top30_bat").head(10).player_id.tolist()
PC = pd.DataFrame(pca.transform(Xs)[:, :3], index=POP, columns=["axis1", "axis2", "axis3"])
prof = PC.reindex(top).assign(name=names.reindex(top).values)
pct = PC.rank(pct=True).reindex(top).round(2).add_suffix("_pct")
log.info("top 10 on the style axes (percentile among all batters):\n" + pd.concat([prof[["name"]], pct], axis=1).to_string())
save_table(pd.concat([PC, PC.rank(pct=True).add_suffix("_pct")], axis=1).assign(name=names.reindex(POP).values).reset_index(), "style_axes_players")
summ = {"population": int(len(POP)), "candidates": CAND, "selected": SEL, "reliability": REL.round(3).to_dict(),
        "real_types_found": bool(T.real_types.fillna(False).any()),
        "positive_control_passes": bool((T_job.p_value <= 0.02).any()),
        "pca_variance": [float(x) for x in ev[:6]]}
save_json(summ, "style_discovery_summary")
log.info(f"positive control (job) beats nulls: {summ['positive_control_passes']} | real playing types found: {summ['real_types_found']}")
